# Robustness and extra-baseline experiments (Colab)
Reviewer comments **R2.6** (larger networks, unseen workloads, changing network) and **R2.5** (comparison with other DRL / optimisation methods).

**Before you start:** upload the *updated* `IoMT_Offloading_Reproducibility_Package.zip` to My Drive (it contains `run_robust.py`, `ppo_agent.py`, the extended simulator and the non-learning results). Use a plain **CPU** runtime.

Two learned methods are run, each trained once in the default environment and then tested in 11 environments (other sizes N = 20 ... 100, burst arrivals, tier-mix shift, heavy tasks, poor channels, 50% capacity drop, all together):

| Block | What | Runs | About (2 cores) |
|---|---|---|---|
| R1 | `dueling_ddqn`, trained at N = 20 and 60 | 20 | about 1.2 h |
| R2 | `ppo` (new baseline), trained at N = 20, 40, 60 | 30 | about 1-2 h |

The non-learning policies and the one-step reference controller (`oracle_onestep`) are already in the package (`raw/robust_nonlearning`).
Everything is resumable. **Start each block ONCE**; use the progress cell afterwards. Do not run R1 and R2 at the same time on 2 cores.

## 1. Mount Drive and unzip (replace the old copy)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PKG = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package"
import os
# keep your earlier raw/ctrl_* results: unzip the new package into a separate folder, then copy the new code over
!mkdir -p /content/new && cd /content/new && unzip -q -o /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package.zip
!cp -r /content/new/IoMT_Offloading_Reproducibility_Package/code/* /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code/
!cp -r /content/new/IoMT_Offloading_Reproducibility_Package/analysis/* /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/analysis/
!mkdir -p /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw && cp -r /content/new/IoMT_Offloading_Reproducibility_Package/raw/robust_nonlearning /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/
!ls /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw
!nproc

## 2. Smoke test (about 2 minutes). It must finish without errors before you start R1 or R2

In [ ]:
!python -c "import torch; print('torch', torch.__version__)"
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && python run_robust.py --arms ppo,dueling_ddqn --N-train 20 --seeds 2 --episodes 5 --n-test 300 --jobs 2 --out /content/smoke_robust 2>&1 | tail -n 15

## 3. Number of parallel jobs (use the number printed by `nproc`)

In [ ]:
JOBS = 2

## Block R1: Dueling Double DQN (proposed agent), robustness tests

In [ ]:
import subprocess
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup python run_robust.py --arms dueling_ddqn --seeds 10 --jobs __JOBS__ --out ../raw/robust_dueling > ../blockR1.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True); print("R1 started")

In [ ]:
!tail -n 3 /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/blockR1.log
import pandas as pd, os
f = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/robust_dueling/per_seed.csv"
if os.path.exists(f): d = pd.read_csv(f); print("finished runs:", d.drop_duplicates(["arm","N_train","seed"]).shape[0], "of 20")
!pgrep -f run_robust.py | wc -l   # 0 = no process running

## Block R2: PPO baseline
Start it only after R1 has finished (finished runs = 20 and process count = 0).

In [ ]:
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup python run_robust.py --arms ppo --seeds 10 --jobs __JOBS__ --out ../raw/robust_ppo > ../blockR2.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True); print("R2 started")

In [ ]:
!tail -n 3 /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/blockR2.log
f = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/robust_ppo/per_seed.csv"
if os.path.exists(f): d = pd.read_csv(f); print("finished runs:", d.drop_duplicates(["arm","N_train","seed"]).shape[0], "of 30")
!pgrep -f run_robust.py | wc -l

## Remove duplicate rows (only needed if a block was started twice or restarted)

In [ ]:
import glob
for folder in sorted(glob.glob("/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/robust_*")):
    f = folder + "/per_seed.csv"; d = pd.read_csv(f); n0 = len(d)
    d.drop_duplicates(["arm","N_train","test_env","seed"], keep="first").to_csv(f, index=False)
    c = folder + "/convergence.csv"
    if os.path.exists(c):
        g = pd.read_csv(c); g.drop_duplicates(["arm","N","seed","episode"], keep="first").to_csv(c, index=False)
    print(folder.split("/")[-1], n0, "->", len(pd.read_csv(f)))

## Analysis (after R1 and R2 have finished)

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && python analysis/robust_analysis.py | tail -n 30

## Collect the results

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && zip -qr /content/drive/MyDrive/robust_results.zip raw/robust_* results/robust_* figures/Fig_robust_* figures/Fig_baselines_default.png blockR1.log blockR2.log 2>/dev/null; ls -la /content/drive/MyDrive/robust_results.zip
from google.colab import files
files.download('/content/drive/MyDrive/robust_results.zip')